In [1]:
!cp -r "/kaggle/input/datasets/krishna26code/tiger-model/Recommender_System" /kaggle/working/
%cd /kaggle/working/Recommender_System

/kaggle/working/Recommender_System


In [2]:
!find /kaggle/input/ -name "*checkpoint*"

/kaggle/input/datasets/krishna26code/rqvae-checkpoint-239999
/kaggle/input/datasets/krishna26code/rqvae-checkpoint-239999/checkpoint_239999.pt


In [3]:
import os
os.makedirs("out/rqvae", exist_ok=True)
!cp /kaggle/input/datasets/krishna26code/rqvae-checkpoint-239999/checkpoint_239999.pt out/rqvae/
!ls -la out/rqvae/

total 8980
drwxr-xr-x 2 root root    4096 Jul 18 14:41 .
drwxr-xr-x 3 root root    4096 Jul 18 14:41 ..
-rw-r--r-- 1 root root 9187270 Jul 18 14:45 checkpoint_239999.pt


In [10]:
!find /kaggle/input/ -iname "*74999*"

/kaggle/input/datasets/krishna26code/checkpoint-74999
/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999


In [11]:
!file /kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999
!ls -la /kaggle/input/datasets/krishna26code/checkpoint-74999/

/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999: directory
total 4
drwxr-xr-x 3 nobody nogroup    0 Jul 18 14:50 .
drwxr-xr-x 5 root   root    4096 Jul 18 14:51 ..
drwxr-xr-x 4 nobody nogroup    0 Jul 18 14:51 checkpoint_74999


In [12]:
!find /kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999 -maxdepth 3

/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999
/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999/.format_version
/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999/.storage_alignment
/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999/data.pkl
/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999/.data
/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999/.data/serialization_id
/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999/version
/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999/byteorder
/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999/data
/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999/data/248
/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999/data/7
/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999/data/135
/kaggle/input/datasets/kris

In [13]:
import zipfile, os

def rezip_checkpoint(src_dir, dst_pt_path):
    base_name = os.path.basename(src_dir.rstrip('/'))
    with zipfile.ZipFile(dst_pt_path, 'w', zipfile.ZIP_STORED) as zf:
        for root, dirs, files in os.walk(src_dir):
            for file in files:
                filepath = os.path.join(root, file)
                arcname = os.path.join(base_name, os.path.relpath(filepath, src_dir))
                zf.write(filepath, arcname)
    print(f"Created {dst_pt_path}")

os.makedirs("out/decoder", exist_ok=True)
rezip_checkpoint(
    "/kaggle/input/datasets/krishna26code/checkpoint-74999/checkpoint_74999",
    "/kaggle/working/Recommender_System/out/decoder/checkpoint_74999.pt"
)

Created /kaggle/working/Recommender_System/out/decoder/checkpoint_74999.pt


In [14]:
import torch
state = torch.load("out/decoder/checkpoint_74999.pt", map_location="cpu", weights_only=False)
print("Decoder iter:", state.get("iter"))

Decoder iter: 74999


In [4]:
!pip install -r requirements.txt --break-system-packages

In [5]:
import torch, torch_geometric, sentence_transformers
print(torch.__version__, torch.cuda.is_available())

2.10.0+cu128 True


In [ ]:
from data.processed import ItemData, RecDataset
item_dataset = ItemData(root="dataset/amazon", dataset=RecDataset.AMAZON, force_process=False, split="beauty")
print("Total unique items in catalog:", len(item_dataset))

In [ ]:
with open("train_rqvae.py", "r") as f:
    content = f.read()

content = content.replace(
    "if __name__ == \"__main__\":",
    "import ast\nif __name__ == \"__main__\":"
)
content = content.replace(
    "args.vae_hidden_dims, args.vae_codebook_size",
    "ast.literal_eval(args.vae_hidden_dims) if isinstance(args.vae_hidden_dims, str) else args.vae_hidden_dims, args.vae_codebook_size"
)
content = content.replace(
    "from torch.optim import AdamW",
    "from torch.optim import AdamW, Adagrad"
)
content = content.replace(
    "optimizer = AdamW(params=model.parameters(), lr=learning_rate, weight_decay=weight_decay)",
    "optimizer = Adagrad(params=model.parameters(), lr=learning_rate)"
)

with open("train_rqvae.py", "w") as f:
    f.write(content)

print("Patched train_rqvae.py: list-parsing fix + Adagrad optimizer")

In [ ]:
!grep -n "Adagrad\|literal_eval" train_rqvae.py

In [ ]:
!python train_rqvae.py --vae_hidden_dims "[512,256,128]" --vae_emb_dim 32 --vae_codebook_size 256 --vae_n_layers 3 --batch_size 1024 --learning_rate 0.4 --commitment_weight 0.25 --iterations 240000 --save_model_every 20000 --save_dir_root out/rqvae/

In [6]:
!ls -la out/rqvae/

total 8980
drwxr-xr-x 2 root root    4096 Jul 18 14:41 .
drwxr-xr-x 3 root root    4096 Jul 18 14:41 ..
-rw-r--r-- 1 root root 9187270 Jul 18 14:45 checkpoint_239999.pt


In [7]:
with open("train_decoder.py", "r") as f:
    content = f.read()
content = content.replace(
    'parser.add_argument("--num_user_bins", default=None)',
    'parser.add_argument("--num_user_bins", type=int, default=None)'
)
with open("train_decoder.py", "w") as f:
    f.write(content)
print("Patched train_decoder.py: num_user_bins ab int hoga")

Patched train_decoder.py: num_user_bins ab int hoga


In [8]:
!grep -n "num_user_bins" train_decoder.py

60:          num_user_bins=None,
141:                                         num_user_bins=num_user_bins)
297:    parser.add_argument("--num_user_bins", type=int, default=None)
314:          args.t5_d_ff, args.t5_num_layers, args.top_k_for_generation, args.should_add_sep_token, args.num_user_bins,


In [15]:
!python train_decoder.py --pretrained_decoder_path out/decoder/checkpoint_74999.pt --pretrained_rqvae_path out/rqvae/checkpoint_239999.pt --vae_hidden_dims "[512,256,128]" --vae_emb_dim 32 --vae_codebook_size 256 --vae_n_layers 3 --batch_size 256 --learning_rate 0.01 --iterations 125000 --num_user_bins 2000 --save_model_every 25000 --save_dir_root out/decoder/

Using existing file P5_data.zip
Extracting dataset/amazon/P5_data.zip
Processing...
Entered Dataset Processing!
config_sentence_transformers.json: 100%|████████| 122/122 [00:00<00:00, 965kB/s]
README.md: 1.78kB [00:00, 6.70MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 387kB/s]
config.json: 1.39kB [00:00, 7.00MB/s]
model.safetensors: 100%|█████████████████████| 219M/219M [00:03<00:00, 64.9MB/s]
tokenizer_config.json: 1.92kB [00:00, 10.9MB/s]
spiece.model: 100%|██████████████████████████| 792k/792k [00:00<00:00, 2.96MB/s]
tokenizer.json: 1.39MB [00:00, 69.2MB/s]
special_tokens_map.json: 1.79kB [00:00, 10.1MB/s]
config.json: 100%|██████████████████████████████| 115/115 [00:00<00:00, 565kB/s]
2_Dense/model.safetensors:   0%|                    | 0.00/2.36M [00:00<?, ?B/s]
2_Dense/pytorch_model.bin:   0%|                    | 0.00/2.36M [00:00<?, ?B/s]

2_Dense/model.safetensors: 100%|███████████| 2.36M/2.36M [00:00<00:00, 5.81MB/s]

2_Dense/pytorch_model.bin

In [20]:
!ls -la out/decoder/

total 291848
drwxr-xr-x 2 root root     4096 Jul 18 22:17 .
drwxr-xr-x 4 root root     4096 Jul 18 14:55 ..
-rw-r--r-- 1 root root 59767275 Jul 18 22:17 checkpoint_124999.pt
-rw-r--r-- 1 root root 59766887 Jul 18 16:27 checkpoint_24999.pt
-rw-r--r-- 1 root root 59766887 Jul 18 17:54 checkpoint_49999.pt
-rw-r--r-- 1 root root 59766887 Jul 18 19:21 checkpoint_74999.pt
-rw-r--r-- 1 root root 59766887 Jul 18 20:49 checkpoint_99999.pt


In [22]:
import torch
from data.processed import ItemData, RecDataset, SeqData
from modules.semids import batch_to, SemanticIdTokenizer
from evaluate.metrics import TopKAccumulator
from modules.model import EncoderDecoderRetrievalModel
from torch.utils.data import DataLoader
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"

item_dataset = ItemData(root="dataset/amazon", dataset=RecDataset.AMAZON,
                         force_process=False, split="beauty")
eval_dataset = SeqData(root="dataset/amazon", dataset=RecDataset.AMAZON,
                        is_train=False, subsample=False, split="beauty")
eval_dataloader = DataLoader(eval_dataset, batch_size=256, shuffle=True)

tokenizer = SemanticIdTokenizer(input_dim=768, output_dim=32,
                                 hidden_dims=[512,256,128], codebook_size=256,
                                 n_layers=3, n_cat_feats=0,
                                 rqvae_weights_path="out/rqvae/checkpoint_239999.pt")
tokenizer.precompute_corpus_ids(item_dataset)
tokenizer.cached_ids = tokenizer.cached_ids.to(device)   # <-- YE NAYI LINE FIX HAI
codebooks = tokenizer.cached_ids[:, :3].cpu()

model = EncoderDecoderRetrievalModel(codebooks=codebooks, num_hierarchies=3,
                                     num_embeddings_per_hierarchy=256,
                                     t5_d_model=128, t5_num_heads=6, t5_d_ff=1024,
                                     t5_num_layers=4, top_k_for_generataion=10,
                                     should_add_sep_token=True, num_user_bins=2000)
model = model.to(device)

checkpoint = torch.load("out/decoder/checkpoint_124999.pt", map_location=device, weights_only=False)
sd = checkpoint["model"]
sd = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in sd.items()}
model.load_state_dict(sd)
model.eval()
print(f"Loaded decoder checkpoint, trained iter: {checkpoint['iter']}")

metrics_accumulator = TopKAccumulator(ks=[1, 5, 10])
with torch.no_grad():
    for batch in tqdm(eval_dataloader, desc="Final Eval"):
        data = batch_to(batch, device)
        tokenized_data = tokenizer(data)
        generated = model.generate_next_sem_id(tokenized_data, top_k=True, temperature=1)
        actual = tokenized_data.sem_ids_fut[:, :3]
        metrics_accumulator.accumulate(actual=actual, top_k=generated.sem_ids)

print("FINAL METRICS:", metrics_accumulator.reduce())

-----Loaded RQVAE Iter 239999-----
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Quantisation Successful
Loaded decoder checkpoint, trained iter: 124999


Final Eval: 100%|██████████| 88/88 [01:07<00:00,  1.30it/s]

FINAL METRICS: {'recall@1': 0.040468631221213615, 'ndcg@1': 0.040468631221213615, 'recall@5': 0.1988999686982963, 'ndcg@5': 0.11838106808745963, 'recall@10': 0.3400706524169387, 'ndcg@10': 0.16386661816375608}
